<a href="https://colab.research.google.com/github/skrixh/enterprise-retail-data-platform/blob/main/python/26MAS1003/data_profiling_pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Profiling — 5 Source Systems
Profiles the raw POS, E-commerce, CRM, Inventory and Supplier sources before they're loaded into the warehouse.

In [12]:
import pandas as pd
import numpy as np
import json
import sqlite3
import xml.etree.ElementTree as ET

pd.set_option("display.max_columns", None)

## Reusable profiling function

In [13]:
def profile_dataset(df, dataset_name):
    """Print a standard profiling report for one dataset."""
    print("=" * 60)
    print(f"DATASET: {dataset_name}")
    print("=" * 60)

    print("\nShape:")
    print(df.shape)

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nData Types:")
    print(df.dtypes)

    missing = df.isnull().sum()
    missing_pct = (df.isnull().mean() * 100).round(2)
    missing_report = pd.DataFrame({"Missing_Count": missing, "Missing_Percentage": missing_pct})
    print("\nMissing Values:")
    print(missing_report)

    print("\nDuplicate Rows:", df.duplicated().sum())

    print("\nDistinct Values per Column:")
    print(df.nunique())

    print("\nNumeric Summary:")
    print(df.describe())

    return {
        "dataset": dataset_name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "missing_total": int(missing.sum()),
        "duplicate_rows": int(df.duplicated().sum()),
    }

## 1. POS Transactions (CSV)

In [14]:
pos = pd.read_csv("pos_transactions.csv")
pos.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [15]:
pos.shape

(10000, 8)

In [16]:
pos.dtypes

,0
InvoiceNo,object
StockCode,object
Description,object
Quantity,int64
InvoiceDate,object
UnitPrice,float64
CustomerID,float64
Country,object


In [17]:
pos.isnull().sum()

,0
InvoiceNo,0
StockCode,0
Description,42
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,2291
Country,0


In [18]:
pos.duplicated().sum()

np.int64(196)

In [19]:
pos["Country"].value_counts()

,count
Country,
United Kingdom,9403
Germany,181
EIRE,109
France,106
Norway,73
Lithuania,34
Italy,24
Japan,16
Australia,14


In [20]:
print("POS negative quantity:", (pos["Quantity"] < 0).sum())
print("POS zero/negative unit price:", (pos["UnitPrice"] <= 0).sum())

POS negative quantity: 130
POS zero/negative unit price: 46


In [21]:
pos_summary = profile_dataset(pos, "POS Transactions")

DATASET: POS Transactions

Shape:
(10000, 8)

Columns:
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

Data Types:
InvoiceNo       object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
UnitPrice      float64
CustomerID     float64
Country         object
dtype: object

Missing Values:
             Missing_Count  Missing_Percentage
InvoiceNo                0                0.00
StockCode                0                0.00
Description             42                0.42
Quantity                 0                0.00
InvoiceDate              0                0.00
UnitPrice                0                0.00
CustomerID            2291               22.91
Country                  0                0.00

Duplicate Rows: 196

Distinct Values per Column:
InvoiceNo       512
StockCode      2015
Description    1981
Quantity        114
InvoiceDate     442
UnitPrice       154
CustomerID      3

## 2. E-commerce Orders (Excel)

In [22]:
ecommerce = pd.read_excel("ecommerce_orders.xlsx")
ecommerce.head()

,order_id,customer_id,product_id,quantity,unit_price,order_date,country
0,O536876,NaN,21313,1,1.66,2010-12-03 11:36:00,United Kingdom
1,O536796,15574.0,20977,2,1.25,2010-12-02 15:46:00,United Kingdom
2,O536544,NaN,48188,1,14.43,2010-12-01 14:32:00,United Kingdom
3,O536799,17228.0,22865,8,2.10,2010-12-02 16:00:00,United Kingdom
4,O536787,17850.0,22752,2,7.65,2010-12-02 15:24:00,United Kingdom


In [23]:
print("Ecommerce missing customer_id:", ecommerce["customer_id"].isnull().sum())
print("Ecommerce negative quantity:", (ecommerce["quantity"] < 0).sum())
print("Ecommerce zero/negative unit price:", (ecommerce["unit_price"] <= 0).sum())

Ecommerce missing customer_id: 1141
Ecommerce negative quantity: 65
Ecommerce zero/negative unit price: 27


In [24]:
ecommerce_summary = profile_dataset(ecommerce, "E-Commerce Orders")

DATASET: E-Commerce Orders

Shape:
(5000, 7)

Columns:
['order_id', 'customer_id', 'product_id', 'quantity', 'unit_price', 'order_date', 'country']

Data Types:
order_id               object
customer_id           float64
product_id             object
quantity                int64
unit_price            float64
order_date     datetime64[ns]
country                object
dtype: object

Missing Values:
             Missing_Count  Missing_Percentage
order_id                 0                0.00
customer_id           1141               22.82
product_id               0                0.00
quantity                 0                0.00
unit_price               0                0.00
order_date               0                0.00
country                  0                0.00

Duplicate Rows: 49

Distinct Values per Column:
order_id        450
customer_id     310
product_id     1583
quantity         93
unit_price      124
order_date      403
country          15
dtype: int64

Numeric Summary:
  

## 3. CRM Customers (JSON)

In [25]:
with open("customers.json", "r") as f:
    crm_data = json.load(f)

crm = pd.DataFrame(crm_data)
crm.head()

,customer_id,country,source_note
0,17850.0,United Kingdom,Derived customer record from UCI Online Retail...
1,13047.0,United Kingdom,Derived customer record from UCI Online Retail...
2,12583.0,France,Derived customer record from UCI Online Retail...
3,13748.0,United Kingdom,Derived customer record from UCI Online Retail...
4,15100.0,United Kingdom,Derived customer record from UCI Online Retail...


In [26]:
crm_summary = profile_dataset(crm, "CRM Customers")

DATASET: CRM Customers

Shape:
(323, 3)

Columns:
['customer_id', 'country', 'source_note']

Data Types:
customer_id    object
country        object
source_note    object
dtype: object

Missing Values:
             Missing_Count  Missing_Percentage
customer_id              0                 0.0
country                  0                 0.0
source_note              0                 0.0

Duplicate Rows: 0

Distinct Values per Column:
customer_id    323
country         15
source_note      1
dtype: int64

Numeric Summary:
       customer_id         country  \
count          323             323   
unique         323              15   
top        13174.0  United Kingdom   
freq             1             297   

                                              source_note  
count                                                 323  
unique                                                  1  
top     Derived customer record from UCI Online Retail...  
freq                                       

## 4. Inventory (XML)

In [27]:
tree = ET.parse("inventory.xml")
root = tree.getroot()

inventory_data = []
for item in root:
    record = {}
    for child in item:
        record[child.tag] = child.text
    inventory_data.append(record)

inventory = pd.DataFrame(inventory_data)

# cast numeric columns back from text (XML gives everything as strings)
for c in ["unit_price"]:
    inventory[c] = pd.to_numeric(inventory[c], errors="coerce")
for c in ["stock_quantity", "reorder_level"]:
    inventory[c] = pd.to_numeric(inventory[c], errors="coerce").astype("Int64")

inventory.head()

,product_id,product_description,unit_price,stock_quantity,reorder_level
0,85123A,WHITE HANGING HEART T-LIGHT HOLDER,2.55,44,95
1,71053,WHITE METAL LANTERN,3.39,386,70
2,84406B,CREAM CUPID HEARTS COAT HANGER,2.75,327,74
3,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,3.39,219,91
4,84029E,RED WOOLLY HOTTIE WHITE HEART.,3.39,216,66


In [28]:
print("Inventory negative stock:", (inventory["stock_quantity"] < 0).sum())
print("Inventory below reorder level:", (inventory["stock_quantity"] < inventory["reorder_level"]).sum())

Inventory negative stock: 0
Inventory below reorder level: 261


In [29]:
inventory_summary = profile_dataset(inventory, "Inventory")

DATASET: Inventory

Shape:
(2015, 5)

Columns:
['product_id', 'product_description', 'unit_price', 'stock_quantity', 'reorder_level']

Data Types:
product_id              object
product_description     object
unit_price             float64
stock_quantity           Int64
reorder_level            Int64
dtype: object

Missing Values:
                     Missing_Count  Missing_Percentage
product_id                       0                0.00
product_description             20                0.99
unit_price                       0                0.00
stock_quantity                   0                0.00
reorder_level                    0                0.00

Duplicate Rows: 0

Distinct Values per Column:
product_id             2015
product_description    1965
unit_price              132
stock_quantity          484
reorder_level            80
dtype: int64

Numeric Summary:
        unit_price  stock_quantity  reorder_level
count  2015.000000          2015.0         2015.0
mean      4.184531

## 5. Supplier Orders (SQL script)
This file is a `CREATE TABLE` + `INSERT` script rather than tabular data, so it's loaded into an in-memory SQLite database and read back out with `pandas.read_sql` instead of a `pd.read_*` call.

In [30]:
with open("supplier_orders.sql", "r") as f:
    sql_script = f.read()

conn = sqlite3.connect(":memory:")
conn.executescript(sql_script)
supplier = pd.read_sql("SELECT * FROM supplier_orders", conn)
conn.close()

supplier.head()

,purchase_order_id,supplier_id,product_id,quantity,unit_cost,order_date,expected_delivery_date,order_status
0,PO000001,SUP056,22819,134,0.32,2026-04-26,2026-04-30,Pending
1,PO000002,SUP043,22985,109,0.33,2026-04-12,2026-04-20,In Transit
2,PO000003,SUP001,22121,124,4.81,2026-01-22,2026-01-29,Delivered
3,PO000004,SUP029,22912,36,3.95,2026-05-06,2026-05-14,In Transit
4,PO000005,SUP007,22706,41,0.21,2026-04-29,2026-05-04,Delivered


In [31]:
print("Supplier order status breakdown:")
print(supplier["order_status"].value_counts())

print("\nDelivery before order date (should be 0):",
      (pd.to_datetime(supplier["expected_delivery_date"]) < pd.to_datetime(supplier["order_date"])).sum())

Supplier order status breakdown:
order_status
In Transit    697
Delivered     661
Pending       642
Name: count, dtype: int64

Delivery before order date (should be 0): 0


In [32]:
supplier_summary = profile_dataset(supplier, "Supplier Orders")

DATASET: Supplier Orders

Shape:
(2000, 8)

Columns:
['purchase_order_id', 'supplier_id', 'product_id', 'quantity', 'unit_cost', 'order_date', 'expected_delivery_date', 'order_status']

Data Types:
purchase_order_id          object
supplier_id                object
product_id                 object
quantity                    int64
unit_cost                 float64
order_date                 object
expected_delivery_date     object
order_status               object
dtype: object

Missing Values:
                        Missing_Count  Missing_Percentage
purchase_order_id                   0                 0.0
supplier_id                         0                 0.0
product_id                          0                 0.0
quantity                            0                 0.0
unit_cost                           0                 0.0
order_date                          0                 0.0
expected_delivery_date              0                 0.0
order_status                       

## Cross-dataset checks
Referential integrity between the transactional tables and their dimension tables.

In [33]:
# cast every product_id to string first - Excel infers product_id as int64
# while the XML/SQL sources keep it as text, which breaks a naive set comparison
pos_products = set(pos["StockCode"].dropna().astype(str).unique())
ecom_products = set(ecommerce["product_id"].dropna().astype(str).unique())
supplier_products = set(supplier["product_id"].dropna().astype(str).unique())
known_products = set(inventory["product_id"].dropna().astype(str).unique())

print("POS product codes NOT in inventory:", len(pos_products - known_products))
print("Ecommerce product codes NOT in inventory:", len(ecom_products - known_products))
print("Supplier product codes NOT in inventory:", len(supplier_products - known_products))

POS product codes NOT in inventory: 0
Ecommerce product codes NOT in inventory: 0
Supplier product codes NOT in inventory: 0


In [34]:
# customer_id formats differ across sources (CRM/POS store trailing ".0", Ecommerce does not) - normalize before comparing
pos_customers = set(pos["CustomerID"].dropna().astype(str).unique())
known_customers = set(crm["customer_id"].dropna().astype(str).unique())
ecom_customers = set(
    ecommerce["customer_id"].dropna().astype(str).apply(lambda x: x if x.endswith(".0") else x + ".0")
)

print("POS customer_ids NOT in CRM:", len(pos_customers - known_customers))
print("Ecommerce customer_ids NOT in CRM (after normalizing format):", len(ecom_customers - known_customers))

POS customer_ids NOT in CRM: 0
Ecommerce customer_ids NOT in CRM (after normalizing format): 0


## Overall summary

In [35]:
summary = pd.DataFrame([pos_summary, ecommerce_summary, crm_summary, inventory_summary, supplier_summary])
summary

,dataset,rows,columns,missing_total,duplicate_rows
0,POS Transactions,10000,8,2333,196
1,E-Commerce Orders,5000,7,1141,49
2,CRM Customers,323,3,0,0
3,Inventory,2015,5,20,0
4,Supplier Orders,2000,8,0,0
